### Feature Preparation

In [21]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import (
    LabelEncoder, minmax_scale, OneHotEncoder, OrdinalEncoder, RobustScaler, StandardScaler, TargetEncoder)

In [22]:
DATA = Path("../data")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 130)

### Load, Choose Rows and Split First

In [23]:
df = pd.read_csv(DATA / "renewals_day3.csv",
                 parse_dates=["renewal_date", "customer_since"])

ren = df[df["renewal_status"] == "Renewed"].copy()
ren["change_pct"] = ren["renewal_acv"] / ren["annual_contract_value"] - 1

print("Renewed Subscriptions :", len(ren))
print(ren["change_pct"].describe().round(2).to_dict())

Renewed Subscriptions : 615
{'count': 615.0, 'mean': 0.05, 'std': 0.19, 'min': -0.35, '25%': -0.08, '50%': 0.06, '75%': 0.19, 'max': 0.6}


In [24]:
train, test = train_test_split(ren, test_size=0.2, random_state=42)

print("Train : ", train.shape, " | Test: ", test.shape)
print("Missing Values in Train :", train[[
      "industry", "employee_count", "sales_channel", "avg_feature_adoption", "nps_score"]].isna().sum().to_dict())

Train :  (492, 32)  | Test:  (123, 32)
Missing Values in Train : {'industry': 8, 'employee_count': 18, 'sales_channel': 8, 'avg_feature_adoption': 21, 'nps_score': 61}


### Encoding Categories

#### 1. Label Encoding

In [25]:
# Label Encoding: each region gets an integer code, assigned in alphabetical order.
# LabelEncoder is meant for target labels; using it on a feature creates a fake numeric order.
le = LabelEncoder()
le.fit(train["region"])
print(dict(zip(le.classes_, le.transform(le.classes_).tolist())))

{'APJC': 0, 'Americas': 1, 'EMEA': 2}


> **What a linear model would conclude:** APJC (0) < Americas (1) < EMEA (2), equally spaced. It treats EMEA as "two steps" past APJC and fits one slope across all three, which is an order and distance that just comes from alphabetical sorting.

#### 2. One-Hot Encoding

In [26]:
# One-Hot Encoding: one 0/1 column per category, so no fake order between categories.
# handle_unknown="ignore" turns an unseen category (LATAM) into all zeros instead of raising an error.
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ohe.fit(train[["segment", "region"]])
print("New Columns :", ohe.get_feature_names_out().tolist())

new_row = pd.DataFrame({"segment": ["SMB"], "region": ["LATAM"]})
print(pd.DataFrame(ohe.transform(new_row), columns=ohe.get_feature_names_out()))

New Columns : ['segment_Enterprise', 'segment_Mid-Market', 'segment_Public Sector', 'segment_SMB', 'region_APJC', 'region_Americas', 'region_EMEA']
   segment_Enterprise  segment_Mid-Market  segment_Public Sector  segment_SMB  region_APJC  region_Americas  region_EMEA
0                 0.0                 0.0                    0.0          1.0          0.0              0.0          0.0


#### 3. Ordinal Encoding

In [27]:
# Ordinal Encoding: we pass the business order explicitly, so Essentials=0 < Advantage=1 < Premier=2.
# Without `categories=`, sklearn would sort alphabetically (Advantage, Essentials, Premier), which is wrong here.
oe = OrdinalEncoder(categories=[["Essentials", "Advantage", "Premier"]])
oe.fit(train[["tier"]])

tiers = pd.DataFrame({"tier": ["Essentials", "Advantage", "Premier"]})
print(tiers.assign(tier_code=oe.transform(tiers)))

         tier  tier_code
0  Essentials        0.0
1   Advantage        1.0
2     Premier        2.0


#### 4. Target Encoding

In [28]:
# Target Encoding: each industry becomes a smoothed mean of change_pct (rare categories are pulled toward the overall mean).
# Missing industries are filled with "Unknown" first so they get their own category.
industry = train[["industry"]].fillna("Unknown")

te = TargetEncoder(target_type="continuous", random_state=42)
te.fit(industry, train["change_pct"])

comparison = pd.DataFrame({"target_encoding": te.encodings_[0]}, index=te.categories_[0])
comparison["raw_mean"] = train["change_pct"].groupby(industry["industry"]).mean()
comparison["rows"] = industry["industry"].value_counts()

print(comparison.round(3))
print("Overall Mean :", round(train["change_pct"].mean(), 3))

                    target_encoding  raw_mean  rows
Education                     0.057     0.057    85
Energy & Utilities            0.100     0.101    67
Financial Services            0.021     0.020    53
Government                    0.076     0.076    44
Healthcare                    0.029     0.029    71
Manufacturing                 0.095     0.097    37
Retail                       -0.001    -0.002    71
Technology                    0.057     0.057    56
Unknown                       0.033     0.032     8
Overall Mean : 0.051


### Scaling Numerical Columns

In [29]:
# Scaling: fit each scaler on the TRAIN values only, then transform train and test with those same train statistics.
# Standard -> mean 0 / std 1, MinMax -> train range [0, 1], Robust -> median 0 / IQR 1 (less affected by outliers).
from sklearn.preprocessing import MinMaxScaler

num_cols = ["annual_contract_value", "avg_seat_utilization"]
scalers = {"Standard": StandardScaler(), "MinMax": MinMaxScaler(), "Robust": RobustScaler()}

rows = []
for name, scaler in scalers.items():
    scaled_train = pd.DataFrame(scaler.fit_transform(train[num_cols]), columns=num_cols)
    scaled_test = pd.DataFrame(scaler.transform(test[num_cols]), columns=num_cols)
    rows.append({
        "scaler": name,
        "train_acv_mean": scaled_train["annual_contract_value"].mean(),
        "train_acv_std": scaled_train["annual_contract_value"].std(ddof=0),  # ddof=0 matches how StandardScaler computes std
        "train_acv_median": scaled_train["annual_contract_value"].median(),
        "test_acv_min": scaled_test["annual_contract_value"].min(),
        "test_acv_max": scaled_test["annual_contract_value"].max(),
        "train_util_min": scaled_train["avg_seat_utilization"].min(),
        "train_util_max": scaled_train["avg_seat_utilization"].max(),
    })

print(pd.DataFrame(rows).set_index("scaler").round(3))

          train_acv_mean  train_acv_std  train_acv_median  test_acv_min  test_acv_max  train_util_min  train_util_max
scaler                                                                                                               
Standard           0.000          1.000            -0.426        -0.571         3.455          -2.851           2.743
MinMax             0.087          0.152             0.022         0.000         0.612           0.000           1.000
Robust             0.705          1.656             0.000        -0.241         6.424          -2.180           2.073


In [30]:
# How squashed is ACV under MinMax? A few very large contracts set the max, so most rows crowd near 0.
# We count the share of training rows that land in the bottom 10% of the [0, 1] range.
acv_minmax = MinMaxScaler().fit_transform(train[["annual_contract_value"]])
print(f"Train ACV Rows Below 0.1 after MinMax : {(acv_minmax < 0.1).mean():.1%}")

Train ACV Rows Below 0.1 after MinMax : 75.8%


### Row-wise Feature Engineering

In [31]:
# add_features(): builds five row-wise features, i.e. each row uses only its own values.
# Because nothing is learned from other rows, it is safe to apply to train and test separately (no leakage).
def add_features(data):
    data = data.copy()
    data["tenure_years"] = (data["renewal_date"] - data["customer_since"]).dt.days / 365.25
    data["log_acv"] = np.log(data["annual_contract_value"])
    data["acv_per_seat"] = data["annual_contract_value"] / data["seats_purchased"]
    data["support_per_100_seats"] = data["support_cases_12m"] / data["seats_purchased"] * 100
    data["had_sev1"] = (data["sev1_cases_12m"] >= 1).astype(int)
    return data


train = add_features(train)
test = add_features(test)
print("Train :", train.shape, " | Test :", test.shape)

Train : (492, 37)  | Test : (123, 37)


In [32]:
# Describe the five new features on train: check their ranges and skew (mean vs median).
# For had_sev1 (0/1), the mean is the share of accounts with at least one Sev-1 case.
new_features = ["tenure_years", "log_acv", "acv_per_seat", "support_per_100_seats", "had_sev1"]
print(train[new_features].describe().T.round(2))

                       count    mean    std     min     25%     50%     75%     max
tenure_years           492.0    6.55   3.35    0.51    3.59    6.62    9.15   12.65
log_acv                492.0   11.46   1.79    7.80   10.11   11.47   12.92   15.24
acv_per_seat           492.0  261.79  88.71  127.08  196.80  240.22  318.11  547.16
support_per_100_seats  492.0    4.32   6.79    0.06    0.48    1.39    4.62   46.67
had_sev1               492.0    0.23   0.42    0.00    0.00    0.00    0.00    1.00


In [33]:
# Correlate each new feature with change_pct using Pearson, on the TRAIN set only (test stays untouched).
# Raw ACV is included so we can compare it against log ACV; rows are sorted by strength, ignoring the sign.
corr = train[new_features + ["annual_contract_value"]].corrwith(train["change_pct"])
print(corr.sort_values(key=abs, ascending=False).round(3))

log_acv                 -0.099
annual_contract_value   -0.085
support_per_100_seats    0.074
acv_per_seat            -0.069
tenure_years             0.040
had_sev1                 0.022
dtype: float64


### Feature Selection for Categorical Columns, Numerical Columns

In [34]:
# Variance Threshold: one-hot encode industry (gaps -> "UNKNOWN"), then drop dummies with variance below 0.02.
# A 0/1 dummy has variance p*(1-p), so 0.02 removes categories present in fewer than ~2% of rows.
industry_ohe = OneHotEncoder(sparse_output=False)
industry_dummies = pd.DataFrame(
    industry_ohe.fit_transform(train[["industry"]].fillna("UNKNOWN")),
    columns=industry_ohe.get_feature_names_out())

vt = VarianceThreshold(threshold=0.02)
vt.fit(industry_dummies)

print(pd.DataFrame({"variance": vt.variances_, "keep": vt.get_support()}, index=industry_dummies.columns).round(4))
print("Dropped :", industry_dummies.columns[~vt.get_support()].tolist())

                             variance   keep
industry_Education             0.1429   True
industry_Energy & Utilities    0.1176   True
industry_Financial Services    0.0961   True
industry_Government            0.0814   True
industry_Healthcare            0.1235   True
industry_Manufacturing         0.0695   True
industry_Retail                0.1235   True
industry_Technology            0.1009   True
industry_UNKNOWN               0.0160  False
Dropped : ['industry_UNKNOWN']


In [35]:
# High correlation: find pairs of numeric features with |r| > 0.85 on train; one of each pair is usually redundant.
# renewal_acv is left out because change_pct is calculated from it (target leakage).
num_features = ["employee_count", "seats_purchased", "discount_pct", "annual_contract_value", "term_months",
                "avg_seat_utilization", "utilization_trend", "avg_feature_adoption", "support_cases_12m",
                "sev1_cases_12m", "nps_score"] + new_features

corr_matrix = train[num_features].corr().abs()
pairs = []
for i, a in enumerate(num_features):
    for b in num_features[i + 1:]:
        if corr_matrix.loc[a, b] > 0.85:
            pairs.append((a, b, round(corr_matrix.loc[a, b], 3)))

print(pd.DataFrame(pairs, columns=["feature_1", "feature_2", "abs_corr"]))

              feature_1              feature_2  abs_corr
0       seats_purchased  annual_contract_value     0.930
1  avg_seat_utilization   avg_feature_adoption     0.896
2        sev1_cases_12m               had_sev1     0.929


In [36]:
# Mutual Information: scores how much knowing a feature tells us about change_pct, including non-linear links (0 = nothing).
# MI cannot handle NaNs, so gaps are filled with the train median first.
X_mi = SimpleImputer(strategy="median").fit_transform(train[num_features])
mi = mutual_info_regression(X_mi, train["change_pct"], random_state=42)
print(pd.Series(mi, index=num_features).sort_values(ascending=False).round(3))

utilization_trend        1.135
log_acv                  0.054
acv_per_seat             0.030
nps_score                0.021
term_months              0.018
avg_seat_utilization     0.014
annual_contract_value    0.008
tenure_years             0.006
seats_purchased          0.005
avg_feature_adoption     0.000
employee_count           0.000
discount_pct             0.000
support_cases_12m        0.000
sev1_cases_12m           0.000
support_per_100_seats    0.000
had_sev1                 0.000
dtype: float64


#### Additional Techniques: Leakage Check, Permutation Importance, Lasso

In [37]:
# Leakage Check: a feature is only usable if it is known BEFORE the renewal is priced.
# open_renewals.csv holds renewals not priced yet, so any column missing there only exists after the outcome.
open_cols = pd.read_csv(DATA / "open_renewals.csv", nrows=0).columns
print("Not Available Before Pricing :", [c for c in df.columns if c not in open_cols])
print("utilization_trend Available Before Pricing :", "utilization_trend" in open_cols)

Not Available Before Pricing : ['csm_health', 'last_csm_note', 'cancellation_notice', 'renewal_status', 'renewal_acv', 'churn_reason']
utilization_trend Available Before Pricing : True


In [38]:
# Permutation Importance: fit a model on train, then shuffle one feature at a time on TEST and measure how much R2 drops.
# A big drop = the model really relies on that feature; ~0 = the feature can be removed without loss.
from sklearn.inspection import permutation_importance

model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LinearRegression())
model.fit(train[num_features], train["change_pct"])
print("Test R2 :", round(r2_score(test["change_pct"], model.predict(test[num_features])), 3))

perm = permutation_importance(model, test[num_features], test["change_pct"], scoring="r2", n_repeats=10, random_state=42)
print(pd.Series(perm.importances_mean, index=num_features).sort_values(ascending=False).round(3))

Test R2 : 0.895
utilization_trend        1.838
avg_seat_utilization     0.109
seats_purchased          0.019
annual_contract_value    0.015
sev1_cases_12m           0.008
support_per_100_seats    0.004
log_acv                  0.003
discount_pct             0.003
had_sev1                 0.002
acv_per_seat             0.002
support_cases_12m        0.001
term_months              0.000
avg_feature_adoption     0.000
nps_score               -0.000
tenure_years            -0.000
employee_count          -0.000
dtype: float64


In [39]:
# Lasso (L1): a linear model that pushes weak or redundant feature coefficients to exactly 0, i.e. it selects features.
# LassoCV picks the penalty strength (alpha) with 5-fold CV on train; features scaled first so coefficients are comparable.
from sklearn.linear_model import LassoCV

lasso = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LassoCV(cv=5, random_state=42))
lasso.fit(train[num_features], train["change_pct"])

coefs = pd.Series(lasso[-1].coef_, index=num_features)
print("Chosen alpha :", round(lasso[-1].alpha_, 5))
print(coefs.sort_values(key=abs, ascending=False).round(4))
print("Dropped by Lasso (coef = 0) :", coefs[coefs == 0].index.tolist())

Chosen alpha : 0.00248
utilization_trend        0.1920
avg_seat_utilization     0.0335
discount_pct             0.0031
acv_per_seat             0.0028
avg_feature_adoption     0.0010
had_sev1                -0.0002
annual_contract_value    0.0000
seats_purchased         -0.0000
term_months              0.0000
employee_count           0.0000
support_cases_12m        0.0000
sev1_cases_12m          -0.0000
tenure_years             0.0000
nps_score                0.0000
log_acv                  0.0000
support_per_100_seats   -0.0000
dtype: float64
Dropped by Lasso (coef = 0) : ['employee_count', 'seats_purchased', 'annual_contract_value', 'term_months', 'support_cases_12m', 'sev1_cases_12m', 'nps_score', 'tenure_years', 'log_acv', 'support_per_100_seats']


### Pipeline with Column Transformer

In [40]:
# preprocess: one ColumnTransformer with three branches, each applying its own steps to its own columns (others are dropped).
# set_output("pandas") keeps column names; this needs sparse_output=False on the OneHotEncoder.
nominal_cols = ["segment", "region", "product", "sales_channel", "industry"]
ordinal_cols = ["tier"]

preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_features),
    ("nom", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore", sparse_output=False)), nominal_cols),
    ("ord", OrdinalEncoder(categories=[["Essentials", "Advantage", "Premier"]]), ordinal_cols),
]).set_output(transform="pandas")

In [41]:
# Full pipeline: preprocess -> LinearRegression, fitted on TRAIN only.
# Every imputer, scaler and encoder inside learns from train, then the same fitted steps are reused on test.
pipe = Pipeline([("preprocess", preprocess), ("model", LinearRegression())])
pipe.fit(train, train["change_pct"])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](37,)","['subscription_id','account_id','account_name',...,'acv_per_seat', 'support_per_100_seats','had_sev1']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,37
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('nom', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically pass

In [43]:
# Inspect: transformed test shape, first six columns, the medians the numeric imputer learnt (from train), and MAE / R2.
# Train vs test metrics side by side show whether the model overfits.
test_transformed = pipe["preprocess"].transform(test)
print("Transformed Test Shape :", test_transformed.shape)
print(test_transformed.iloc[:, :6].head().round(3))

num_imputer = pipe["preprocess"].named_transformers_["num"][0]
print("Medians Learnt by the Numeric Imputer :")
print(pd.Series(num_imputer.statistics_, index=num_features).round(3))

for name, data in [("Train", train), ("Test", test)]:
    pred = pipe.predict(data)
    print(f"{name} MAE : {mean_absolute_error(data['change_pct'], pred):.4f} | R2 : {r2_score(data['change_pct'], pred):.3f}")

Transformed Test Shape : (123, 38)
     num__employee_count  num__seats_purchased  num__discount_pct  num__annual_contract_value  num__term_months  \
332               -0.254                -0.045              1.172                      -0.206            -0.665   
481               -0.395                 0.475              0.658                       0.358             1.503   
571               -0.468                -0.598             -0.543                      -0.571            -0.665   
789               -0.339                -0.391             -1.186                      -0.283            -0.665   
179                0.792                -0.479              1.844                      -0.517            -0.665   

     num__avg_seat_utilization  
332                      0.843  
481                      1.307  
571                     -1.135  
789                     -1.188  
179                      1.036  
Medians Learnt by the Numeric Imputer :
employee_count             869.000
s